In [5]:
# TODO:
# Get KD working

In [6]:
# Import packages
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, TensorDataset
from torchvision import transforms
import numpy as np
import random
import os
import json
from sklearn.model_selection import train_test_split
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn
# import tqdm.notebook as tqdm
from tqdm import tqdm
import torch
import wfdb
import math
import importlib.metadata
import json
import logging
import os
import re
import tempfile
import time
import ast
from pathlib import Path
from typing import Any, Callable, Dict, List, Literal, Optional, Tuple, Type, TypeVar, Union, Collection
from sklearn.metrics import roc_auc_score, accuracy_score, precision_recall_fscore_support
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc, roc_auc_score, precision_recall_fscore_support, accuracy_score
from sklearn.metrics import accuracy_score, roc_auc_score, precision_recall_fscore_support, confusion_matrix
from sklearn.preprocessing import label_binarize
import numpy as np
from tqdm import tqdm
import torch


import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import seaborn as sns
import sklearn
import torch


# Ensure reproducibility
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":16:8"
random.seed(0)
np.random.seed(0)
torch.manual_seed(0)
torch.cuda.manual_seed(0)
torch.cuda.manual_seed_all(0)  # For multi-GPU
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
torch.use_deterministic_algorithms(True)

In [7]:
# Step 1: Load in predictions and indices from json file
with open('../../Cardiac-Death-Prediction/LLM/subject-info-cleaned-with-prognosis-D-Llama3B_dmis-lab_biobert-base-cased-v1.1_predictions.json', 'r') as file:
    data = json.load(file)
train_predictions = data['train_predictions']
val_predictions = data['val_predictions']

with open('../../Cardiac-Death-Prediction/LLM/split_indices.json', 'r') as file:
    data = json.load(file)
train_idx = data['train_indices']
val_idx = data['val_indices']

# Combine predictions and indices
LMLLM_train = pd.DataFrame({
    'train_predictions': train_predictions,
    'train_indices': train_idx
})
LMLLM_test = pd.DataFrame({
    'test_predictions': val_predictions, 
    'test_indices': val_idx
})

In [8]:
# Step 2: Find patients with 2-lead ECGs and remove them
from tqdm import tqdm
import wfdb

two_lead_patients = []
missing_patients = []

for i in tqdm(range(1, 1074)):
    patient = f"P{str(i).zfill(4)}"
    record_name = ecg_dir = f"/projects/bdlo/music-sudden-cardiac-death/Holter_ECG/{patient}"

    try:
        # Try loading the signal and header info
        record = wfdb.rdrecord(record_name)
        signal = record.p_signal

        # Check for 2-lead patients
        if signal.shape[1] != 3:
            two_lead_patients.append(i)

    except FileNotFoundError:
        missing_patients.append(i)
        continue
    except Exception as e:
        print(f"Skipping {patient} due to unexpected error: {e}")
        missing_patients.append(i)
        continue

# You can print or save the lists
print("2-lead patients:", two_lead_patients)
print("Missing patients:", missing_patients)



100%|██████████| 1073/1073 [15:28<00:00,  1.16it/s]

2-lead patients: [70, 125, 174, 247, 258, 302, 305, 310, 384, 421, 424, 425, 427, 448, 460, 462, 466, 489, 490, 493, 494, 495, 588, 592, 628, 661, 674, 706, 738, 792, 813, 814, 829, 831, 832, 898, 962, 1073]
Missing patients: [109, 117, 123, 144, 180, 185, 245, 246, 250, 251, 300, 308, 313, 368, 369, 370, 383, 385, 393, 396, 397, 410, 429, 431, 432, 433, 434, 439, 440, 445, 446, 477, 483, 491, 553, 563, 565, 570, 576, 584, 605, 689, 690, 703, 717, 718, 722, 742, 793, 794, 809, 830, 851, 858, 869, 881, 890, 897, 899, 908, 916, 917, 918, 919, 920, 921, 927, 929, 934, 936, 944, 948, 950, 952, 959, 963, 965, 966, 975, 979, 985, 986, 987, 989, 990, 993, 994, 995, 1000, 1003, 1004, 1005, 1006, 1007, 1008, 1009, 1010, 1012, 1014, 1015, 1022, 1024, 1025, 1028, 1033, 1034, 1035, 1036, 1037, 1038, 1039, 1040, 1041, 1042, 1043, 1044, 1045, 1046, 1047, 1048, 1049, 1050, 1051, 1052, 1053, 1055, 1059, 1060, 1061, 1062, 1063, 1067, 1068, 1069, 1070, 1071, 1072]


In [9]:
# Step 3: Find corresponding patient IDs and indices in subject-info-cleaned-with-prognosis-D-Llama3B.csv
# As long as it is plan D, it is fine
df_prognosis = pd.read_csv("../../Cardiac-Death-Prediction/Data/subject-info-cleaned-with-prognosis-D-Llama3B.csv")

# Convert patient ID arrays to sets for faster lookup
excluded_patients = set(two_lead_patients + missing_patients)

# Filter rows where P#### is not in excluded list
df_prognosis = df_prognosis[~df_prognosis['Patient ID'].isin([f"P{str(i).zfill(4)}" for i in excluded_patients])]

# Map labels to integers
label_map = {"survivor": 0, "sudden cardiac death": 1, "pump failure death": 2}
df_prognosis['Outcome'] = df_prognosis['Outcome'].map(label_map)

In [10]:
# Step 4: Perform train/test split, then extract ground truths and teacher predictions
# Filter rows for training and test sets
train_df = df_prognosis[df_prognosis['Patient ID'].isin(train_idx)]
train_df = train_df.sort_values(by = 'Patient ID')
test_df = df_prognosis[df_prognosis['Patient ID'].isin(val_idx)]
test_df = test_df.sort_values(by = 'Patient ID')

# Get ground truths
train_truth = train_df['Outcome']
test_truth = test_df['Outcome']

# Get teacher predictions
teacher_train = pd.merge(LMLLM_train, train_df, left_on = 'train_indices', right_on = 'Patient ID', how = 'inner')
teacher_train = teacher_train.sort_values(by = 'Patient ID')
teacher_train_truth = teacher_train['Outcome']
teacher_test = pd.merge(LMLLM_test, test_df, left_on = 'test_indices', right_on = 'Patient ID', how = 'inner')
teacher_test = teacher_test.sort_values(by = 'Patient ID')
teacher_test_truth = teacher_test['Outcome']

# Get Patient IDs
train_patient_ids = teacher_train['Patient ID']
test_patient_ids = teacher_test['Patient ID']

In [11]:
# Step 5: Load in ECG data
class Jitter(object):
    def __init__(self, sigma=0.03):
        self.sigma = sigma

    def jitter(self, x, sigma):
        # Jitter is added to every point in the time series data, so no change is needed based on channel_first
        return x + np.random.normal(loc=0., scale=sigma, size=x.shape)
    
    def __call__(self, x):
        return self.jitter(x, self.sigma)


class Scaling(object):
    def __init__(self, sigma=0.1, channel_first=False):
        self.sigma = sigma
        self.channel_first = channel_first

    def scaling(self, x, sigma):
        factor = np.random.normal(loc=1., scale=sigma, size=(1, x.shape[1]))  # Shape: (1, C)
        return np.multiply(x, factor)
    
    def __call__(self, x):
        return self.scaling(x, self.sigma)
    
class ToTensor(object):
    """Convert ndarrays in sample to Tensors."""

    def __call__(self, sample):
        sample = torch.Tensor(sample)
        return sample
    

class TwoCropTransform:
    """Create two crops of the same image"""
    def __init__(self, transform):
        self.transform = transform

    def __call__(self, x):
        return [self.transform(x), self.transform(x)]
    
def normalize(signal):
    mean = signal.mean(dim=-1, keepdim=True)  # Mean along time axis
    std = signal.std(dim=-1, keepdim=True)
    return (signal - mean) / (std + 1e-6)  # Avoid division by zero

class HolterECGLoader(Dataset):
    def __init__(self, csv_file, ecg_dir, chunk_size = 1000, target_length=12000000):
        """
        ECG Dataset that returns one 5-second segment per call and moves sequentially to the next patient.

        Args:
            csv_file (str): Path to the CSV file containing metadata.
            ecg_dir (str): Path to the directory containing ECG records.
            segment_length (int): Length of ECG segments in seconds.
            augmentation (bool): Whether to apply augmentation.
        """
        super().__init__()
        self.dataset = pd.read_csv(csv_file, index_col=0)
        self.label_dict = {0: 0, 1: 1, 2: 2}
        self.ecg_dir = ecg_dir
        
        self.chunk_size = chunk_size
        self.target_length = target_length 

        self.transform = transforms.Compose([
            ToTensor(),
        ])

    def __len__(self):
        """
        Returns the number of patients since the dataloader will iterate per patient, not per segment.
        """
        return len(self.dataset)


    def mean_impute(self, ecg_signal):
        """Impute missing values in a 3-channel ECG using mean imputation."""
        mask = torch.isnan(ecg_signal)  # Find missing values
        mean_values = torch.nanmean(ecg_signal, dim=0, keepdim=True)  # Compute mean per channel
        ecg_signal[mask] = mean_values.expand_as(ecg_signal)[mask]  # Replace NaNs with per-channel mean
        return ecg_signal

    
    def load_ecg(self, ecg_filename): 
        # Load patient ECG file
        record = wfdb.rdrecord(ecg_filename)
        signal = record.p_signal  
        fs = record.fs  

        # Trim first and last 30 seconds
        trim_samples = fs * 30
        signal = signal[trim_samples:-trim_samples]
    
        # Ensure the signal is at least target_length
        if signal.shape[0] > self.target_length:
            signal = signal[:self.target_length, :]
        elif signal.shape[0] < self.target_length:
            # pad_length = self.target_length - signal.shape[1]
            # signal = np.pad(signal, ((0, 0), (0, pad_length)), mode='constant')
            pad_length = self.target_length - signal.shape[0]
            signal = np.pad(signal, ((0, pad_length), (0, 0)), mode='constant')
        
        return signal, fs
        

    def __getitem__(self, idx):
        """
        Returns one 5-second segment from the current patient's ECG.
        Moves to the next patient after all segments of the current one are processed.
        """
        if torch.is_tensor(idx):
            idx = idx.tolist()

        #load
        # ecg_filename = os.path.join(self.ecg_dir, self.dataset.iloc[idx, 0])
        ecg_filename = os.path.join(self.ecg_dir, self.dataset['Patient ID'][idx])
        # label = self.label_dict[self.dataset.iloc[idx, 4]]
        label = self.label_dict[self.dataset['Outcome'][idx]]
        signal, fs = self.load_ecg(ecg_filename)
        
        #transform
        signal = self.transform(signal)
        signal = self.mean_impute(signal)
        signal = normalize(signal)
    
        
        #from (seq_length, 3) to (3, num chunks, chunk size) NONOVERLAPPING SEGMENTS
        signal = signal.unfold(dimension=0, size=self.chunk_size, step=self.chunk_size - 100)
        return signal, label 


In [12]:
csv_path = "../../Cardiac-Death-Prediction/Data/subject-info-cleaned-with-prognosis-D-Llama3B-Outcome.csv"
# csv_path = "/projects/bdlo/music-sudden-cardiac-death/subject-info-cleaned.csv"
ecg_dir = "/projects/bdlo/music-sudden-cardiac-death/Holter_ECG"

holterecg_dataset = HolterECGLoader(csv_file=csv_path, ecg_dir=ecg_dir)

In [13]:
# Step 6: Load in model
Floats = Union[float, List[float]]

class ECG_downsampler(nn.Module):
    def __init__(self, output_segments=1000):
        """
        ARGS:
            output_segments: Number of segments to retain after downsampling.
        """
        super().__init__()

        #aggressive downsampling
        #instance norm samples each segment independently, ensuring no bleeding
        self.conv1 = nn.Conv1d(in_channels=3, out_channels=16, kernel_size=25, stride=20, padding=12)
        self.pool1 = nn.MaxPool1d(kernel_size=10, stride=10)  # Reduce faster
        self.norm1 = nn.InstanceNorm1d(16)  # Prevents batch mixing

        self.conv2 = nn.Conv1d(in_channels=16, out_channels=3, kernel_size=5, stride=1, padding=2)
        self.norm2 = nn.InstanceNorm1d(3)

        self.adaptive_pool = nn.AdaptiveAvgPool1d(1)

    def forward(self, x):
        """
        ARGS:
            x: Input ECG signal shape (batch, 3, num_segments, temporal)
        
        RETURNS: (batch, 3, output_segments)
        """

        batch_size, num_leads, num_segments, time_dim = x.shape

        # flattens x, so that x is all segments in the batch
        x = x.reshape(batch_size * num_segments, num_leads, time_dim)  # Merge batch & segments

        # downsample time dim, using adaptive pooling
        x = self.pool1(self.norm1(self.conv1(x)).relu())  
        x = self.norm2(self.conv2(x)).relu()  

        # force the time dim to 1, so it is (batchsize * num_segments, 3, 1)
        x = self.adaptive_pool(x)  

        # remove last dim
        x = x.squeeze(-1) 

        num_segments = x.shape[0] // batch_size  
        #reshape to get original sizes
        x = x.contiguous().view(batch_size, num_segments, num_leads)  
        
        
        # switch axis for input to encoder(batch, 3, num_segments)
        x = x.permute(0, 2, 1)  

        return x

class XResNet1D(nn.Module):
    
    #INPUT IS (batch, 3, ecg_length)
    def __init__(self, in_channels=3, num_classes=3, layers=[2, 2, 2, 2]):
        super(XResNet1D, self).__init__()

        self.inplanes = 64
        self.conv1 = nn.Conv1d(in_channels,out_channels=64, kernel_size=7, stride=2, padding=3, bias=False)
        self.bn1 = nn.BatchNorm1d(64)
        self.relu = nn.ReLU(inplace=True)
        self.maxpool = nn.MaxPool1d(kernel_size=3, stride=2, padding=1)

        # Define ResNet blocks
        self.layer1 = self._make_layer(64, layers[0])
        self.layer2 = self._make_layer(128, layers[1], stride=2)
        self.layer3 = self._make_layer(256, layers[2], stride=2)
        self.layer4 = self._make_layer(512, layers[3], stride=2)

        self.avgpool = nn.AdaptiveAvgPool1d(1)
        self.fc = nn.Linear(512, num_classes)

    def _make_layer(self, planes, blocks, stride=1):
        layers = []
        layers.append(self._residual_block(self.inplanes, planes, stride))
        self.inplanes = planes
        for _ in range(1, blocks):
            layers.append(self._residual_block(planes, planes))
        return nn.Sequential(*layers)

    def _residual_block(self, in_planes, out_planes, stride=1):
        return nn.Sequential(
            nn.Conv1d(in_planes, out_planes, kernel_size=3, stride=stride, padding=1, bias=False),
            nn.BatchNorm1d(out_planes),
            nn.ReLU(inplace=True),
            nn.Conv1d(out_planes, out_planes, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm1d(out_planes)
        )

    def forward(self, x):
        x = self.conv1(x)  # Initial Conv Layer
        x = self.bn1(x)
        x = self.relu(x)
        x = self.maxpool(x)

        x = self.layer1(x)  # Residual Blocks
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)

        x = self.avgpool(x)  # Global Pooling
        x = torch.flatten(x, 1)
        x = self.fc(x)  # Classification Head
        return x

class ECG_Encoder(nn.Module):
    def __init__(self, ecg_downsampler = ECG_downsampler, encoder_model = XResNet1D):
        super().__init__()
        self.ecg_downsampler = ecg_downsampler
        self.encoder_model = encoder_model

    def forward(self, x):
        """
        ARGS:
            x: Input ECG of shape (batch_size, num_segments, 3, temporal)
        RETURNS: num_classes
        """
        
        #input is (batch, num_seg, 3, time)
        batch_size, num_segments, num_leads, time_dim = x.shape  # Unpack shape

        # process each ECG segment independently through downsampler
        x = x.permute(0, 2, 1, 3)  # move num_leads forward --> (batch, 3, num_segments, temporal)
        
        #input should be (batch, 3, num_segments, temporal)
        x = self.ecg_downsampler(x)  

        # resnet
        x = self.encoder_model(x) 

        return F.log_softmax(x, dim = 1)

In [14]:
# Step 7: Train
def one_train_epoch(model, dataloader, criterion, optimizer, scheduler, device, clip_grad=1.0):
    model.train()  # Set model to training mode
    total_loss = 0
    all_preds = []
    all_labels = []
    all_probs = []

    # Progress bar
    pbar = tqdm(enumerate(dataloader), total=len(dataloader), desc="Training", leave=True)

    for batch_idx, (x, y) in pbar:
        x, y = x.to(device), y.to(device)  # Move to GPU if available
        
        optimizer.zero_grad()  
        outputs = model(x)  # Shape: (batch_size, num_classes)
        
        loss = criterion(outputs, y)
        total_loss += loss.item()

        preds = torch.argmax(outputs, dim=1)  
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(y.cpu().numpy())
        
        probs = torch.exp(outputs)
        all_probs.extend(probs.detach().cpu().numpy())

        loss.backward()  
        torch.nn.utils.clip_grad_norm_(model.parameters(), clip_grad)
        optimizer.step() 

        pbar.set_postfix({"Loss": f"{loss.item():.4f}"})

    # Compute final metrics
    avg_loss = total_loss / len(dataloader)
    accuracy = accuracy_score(all_labels, all_preds)
    auc = roc_auc_score(all_labels, all_probs, multi_class="ovr", average="macro")
    precision, recall, f1, _ = precision_recall_fscore_support(all_labels, all_preds, average="macro", zero_division=1)

    scheduler.step(avg_loss)
    
    print(f"Train Loss: {avg_loss:.4f}")
    print(f"Accuracy: {accuracy:.2f}")
    print(f"Precision: {precision:.4f}, Recall: {recall:.4f}, F1-score: {f1:.4f}")
    print(f"AUC Score: {auc:.4f}\n")

    return avg_loss, accuracy, precision, recall, f1

def one_test_epoch(model, dataloader, criterion, device, save_path="auroc_plot.png", cm_save_path="confusion_matrix.png"):
    model.eval()
    total_loss = 0
    all_preds = []
    all_labels = []
    all_probs = []

    # Progress bar
    pbar = tqdm(enumerate(dataloader), total=len(dataloader), desc="Testing", leave=True)

    with torch.no_grad():
        for batch_idx, (x, y) in pbar:
            x, y = x.to(device), y.to(device)
            print(y)
            outputs = model(x)
            loss = criterion(outputs, y)
            total_loss += loss.item()

            preds = torch.argmax(outputs, dim=1)
            probs = torch.softmax(outputs, dim=1)  # Better than exp for logits

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(y.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())

            pbar.set_postfix({"Loss": f"{loss.item():.4f}"})

    all_labels = np.array(all_labels)
    all_probs = np.array(all_probs)

    # Compute metrics
    avg_loss = total_loss / len(dataloader)
    accuracy = accuracy_score(all_labels, all_preds)
    auc_score = roc_auc_score(all_labels, all_probs, multi_class="ovr", average="macro")
    precision, recall, f1, _ = precision_recall_fscore_support(all_labels, all_preds, average="macro", zero_division=1)

    print(f"\nTest Loss: {avg_loss:.4f}")
    print(f"Accuracy: {accuracy:.2f}")
    print(f"Precision: {precision:.4f}, Recall: {recall:.4f}, F1-score: {f1:.4f}")
    print(f"AUC Score: {auc_score:.4f}\n")

    # --- AUROC Plot ---
    n_classes = all_probs.shape[1]
    y_true_bin = label_binarize(all_labels, classes=np.arange(n_classes))

    fpr = dict()
    tpr = dict()
    roc_auc = dict()

    plt.figure(figsize=(8, 6))
    class_names = ['Survivor', 'Sudden Cardiac Death', 'Pump Failure Death']
    for i in range(n_classes):
        fpr[i], tpr[i], _ = roc_curve(y_true_bin[:, i], all_probs[:, i])
        roc_auc[i] = auc(fpr[i], tpr[i])
        plt.plot(fpr[i], tpr[i], label=f"{class_names[i]} (AUC = {roc_auc[i]:.2f})")

    plt.plot([0, 1], [0, 1], 'k--', label='Random Guess')
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title("ECG Module Only Multi-Class AUROC")
    plt.legend(loc="lower right")
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(save_path)
    plt.close()

    # --- Confusion Matrix ---
    cm = confusion_matrix(all_labels, all_preds)
    cm_df = pd.DataFrame(cm, index=class_names, columns=class_names)

    plt.figure(figsize=(8, 6))
    sns.heatmap(cm_df, annot=True, fmt="d", cmap="Blues", cbar=False)
    plt.title("ECG Module Only Confusion Matrix")
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.tight_layout()
    plt.savefig(cm_save_path)
    plt.close()

    return avg_loss, accuracy, precision, recall, f1, auc_score

In [15]:
df_ids = pd.read_csv("../../Cardiac-Death-Prediction/Data/subject-info-cleaned-with-prognosis-D-Llama3B-Outcome.csv")
train_idx = df_ids[df_ids['Patient ID'].isin(train_patient_ids)].index.tolist()
val_idx = df_ids[df_ids['Patient ID'].isin(test_patient_ids)].index.tolist()

In [16]:
train_patient_ids

316    P0001
525    P0002
576    P0004
459    P0005
424    P0007
       ...  
45     P1057
453    P1058
224    P1064
478    P1065
211    P1066
Name: Patient ID, Length: 648, dtype: object

In [17]:
batch_size = 3
seq_length = 1000  
num_classes = 3
lr = 0.001
weight_decay = 1e-4
target_length = 6000000
num_epochs = 10

device = "cuda:0"
ecg_downsampler = ECG_downsampler()
encoder_model = XResNet1D()
model = ECG_Encoder(ecg_downsampler = ecg_downsampler, encoder_model = encoder_model).to(device)

criterion = torch.nn.NLLLoss()
optimizer = optim.AdamW(model.parameters(), lr = lr, weight_decay = weight_decay) 
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=2, verbose=True)

#train loader
train_sampler = torch.utils.data.sampler.SubsetRandomSampler(train_idx)
holterecg_dataset = HolterECGLoader(csv_file=csv_path, ecg_dir = ecg_dir, chunk_size = seq_length, target_length = target_length)
train_loader = DataLoader(holterecg_dataset, batch_size= batch_size, num_workers=2, sampler=train_sampler)

val_sampler = torch.utils.data.sampler.SubsetRandomSampler(val_idx)
val_loader = DataLoader(holterecg_dataset, batch_size = batch_size, num_workers=2, sampler=val_sampler)

/u/sswee/miniconda3/lib/python3.12/site-packages/torch/optim/lr_scheduler.py:28: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn("The verbose parameter is deprecated. Please use get_last_lr() "


In [18]:
for i in range(num_epochs):
    print("Epoch: %s" %i)
    one_train_epoch(model = model, 
                    dataloader = train_loader, 
                    criterion = criterion, 
                    optimizer = optimizer, 
                    scheduler = scheduler, 
                    device = device, 
                    clip_grad=1.0)
    
    # one_test_epoch(model = model, 
    #                      dataloader = train_loader, 
    #                      criterion = criterion,  
    #                      device = device)
    print("-------------------------")

Epoch: 0


/u/sswee/miniconda3/lib/python3.12/site-packages/torch/nn/modules/conv.py:306: UserWarning: Plan failed with a cudnnException: CUDNN_BACKEND_EXECUTION_PLAN_DESCRIPTOR: cudnnFinalize Descriptor Failed cudnn_status: CUDNN_STATUS_NOT_SUPPORTED (Triggered internally at /opt/conda/conda-bld/pytorch_1712609048481/work/aten/src/ATen/native/cudnn/Conv_v8.cpp:919.)
  return F.conv1d(input, weight, bias, self.stride,
Training: 100%|██████████| 216/216 [11:47<00:00,  3.28s/it, Loss=1.7739]

Train Loss: 0.9644
Accuracy: 0.69
Precision: 0.2958, Recall: 0.3145, F1-score: 0.2980
AUC Score: 0.4576

-------------------------
Epoch: 1



Training: 100%|██████████| 216/216 [12:18<00:00,  3.42s/it, Loss=1.2777]


Train Loss: 0.8242
Accuracy: 0.76
Precision: 0.9213, Recall: 0.3333, F1-score: 0.2887
AUC Score: 0.4964

-------------------------
Epoch: 2


...

Train Loss: 0.7779
Accuracy: 0.76
Precision: 0.9213, Recall: 0.3333, F1-score: 0.2887
AUC Score: 0.5047

-------------------------
Epoch: 8


Training: 100%|██████████| 216/216 [09:25<00:00,  2.62s/it, Loss=1.1050]

Train Loss: 0.8909
Accuracy: 0.76
Precision: 0.9213, Recall: 0.3333, F1-score: 0.2887
AUC Score: 0.4879

-------------------------
Epoch: 9



Training: 100%|██████████| 216/216 [09:23<00:00,  2.61s/it, Loss=0.1136]

Train Loss: 0.8174
Accuracy: 0.76
Precision: 0.9213, Recall: 0.3333, F1-score: 0.2887
AUC Score: 0.4810

-------------------------


In [19]:
!nvidia-smi

Sun Apr  6 19:40:50 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.144.03             Driver Version: 550.144.03     CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A40                     On  |   00000000:85:00.0 Off |                    0 |
|  0%   41C    P0             80W /  300W |    1445MiB /  46068MiB |     28%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [20]:
one_test_epoch(model = model, 
             dataloader = val_loader, 
             criterion = criterion,  
             device = device)

Testing:   2%|▏         | 1/55 [00:06<05:33,  6.17s/it, Loss=0.0951]

tensor([0, 0, 0], device='cuda:0')
tensor([0, 0, 0], device='cuda:0')


Testing:   7%|▋         | 4/55 [00:11<02:03,  2.42s/it, Loss=0.1000]

tensor([0, 0, 0], device='cuda:0')
tensor([0, 0, 0], device='cuda:0')


Testing:   9%|▉         | 5/55 [00:17<02:52,  3.45s/it, Loss=0.0808]

...

Testing:  96%|█████████▋| 53/55 [02:28<00:05,  2.68s/it, Loss=1.0937]

tensor([0, 0, 0], device='cuda:0')
tensor([0, 2, 0], device='cuda:0')


Testing: 100%|██████████| 55/55 [02:33<00:00,  2.80s/it, Loss=1.0494]

tensor([2, 0, 0], device='cuda:0')

Test Loss: 0.7578
Accuracy: 0.78
Precision: 0.9273, Recall: 0.3333, F1-score: 0.2925
AUC Score: 0.4212



(0.7577595175667242,
 0.7818181818181819,
 0.9272727272727272,
 0.3333333333333333,
 0.2925170068027211,
 np.float64(0.4212460507514855))

In [25]:
def one_test_epoch(
    model, dataloader, criterion, device,
    save_path="ECGOnly_auroc_plot.png", cm_save_path="ECGOnly_confusion_matrix.png",
    roc_csv_path="ECGOnly_auroc_data.csv"
):
    model.eval()
    total_loss = 0
    all_preds = []
    all_labels = []
    all_probs = []

    pbar = tqdm(enumerate(dataloader), total=len(dataloader), desc="Testing", leave=True)

    with torch.no_grad():
        for batch_idx, (x, y) in pbar:
            x, y = x.to(device), y.to(device)
            outputs = model(x)
            loss = criterion(outputs, y)
            total_loss += loss.item()

            preds = torch.argmax(outputs, dim=1)
            probs = torch.softmax(outputs, dim=1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(y.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())

            pbar.set_postfix({"Loss": f"{loss.item():.4f}"})

    all_labels = np.array(all_labels)
    all_probs = np.array(all_probs)

    avg_loss = total_loss / len(dataloader)
    accuracy = accuracy_score(all_labels, all_preds)
    auc_score = roc_auc_score(all_labels, all_probs, multi_class="ovr", average="macro")
    precision, recall, f1, _ = precision_recall_fscore_support(all_labels, all_preds, average="macro", zero_division=1)

    print(f"\nTest Loss: {avg_loss:.4f}")
    print(f"Accuracy: {accuracy:.2f}")
    print(f"Precision: {precision:.4f}, Recall: {recall:.4f}, F1-score: {f1:.4f}")
    print(f"AUC Score: {auc_score:.4f}\n")

    # --- AUROC Plot & Save Data ---
    n_classes = all_probs.shape[1]
    y_true_bin = label_binarize(all_labels, classes=np.arange(n_classes))

    fpr = dict()
    tpr = dict()
    roc_auc = dict()
    auroc_data = []

    plt.figure(figsize=(8, 6))
    class_names = ['Survivor', 'Sudden Cardiac Death', 'Pump Failure Death']
    for i in range(n_classes):
        fpr[i], tpr[i], _ = roc_curve(y_true_bin[:, i], all_probs[:, i])
        roc_auc[i] = auc(fpr[i], tpr[i])

        # Save FPR/TPR data for CSV
        for f, t in zip(fpr[i], tpr[i]):
            auroc_data.append({
                "class": class_names[i],
                "fpr": f,
                "tpr": t,
                "auc": roc_auc[i]
            })

        plt.plot(fpr[i], tpr[i], label=f"{class_names[i]} (AUC = {roc_auc[i]:.2f})")

    # Save AUROC data to CSV
    auroc_df = pd.DataFrame(auroc_data)
    auroc_df.to_csv(roc_csv_path, index=False)

    plt.plot([0, 1], [0, 1], 'k--', label='Random Guess')
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title("ECG Module Only Multi-Class AUROC")
    plt.legend(loc="lower right")
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(save_path)
    plt.close()

    # --- Confusion Matrix ---
    cm = confusion_matrix(all_labels, all_preds)
    cm_df = pd.DataFrame(cm, index=class_names, columns=class_names)

    plt.figure(figsize=(8, 6))
    sns.heatmap(cm_df, annot=True, fmt="d", cmap="Blues", cbar=False)
    plt.title("ECG Module Only Confusion Matrix")
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.tight_layout()
    plt.savefig(cm_save_path)
    plt.close()

    return avg_loss, accuracy, precision, recall, f1, auc_score

In [26]:
one_test_epoch(model = model, 
             dataloader = val_loader, 
             criterion = criterion,  
             device = device)

Testing: 100%|██████████| 55/55 [02:19<00:00,  2.54s/it, Loss=0.0855]



Test Loss: 0.7578
Accuracy: 0.78
Precision: 0.9273, Recall: 0.3333, F1-score: 0.2925
AUC Score: 0.4212



(0.7577595147219571,
 0.7818181818181819,
 0.9272727272727272,
 0.3333333333333333,
 0.2925170068027211,
 np.float64(0.4212460507514855))

In [27]:
def plot_auroc_from_csv(csv_path, save_path="ECGOnly_auroc_from_csv.png", title="AUROC Curve"):
    # Load AUROC data
    df = pd.read_csv(csv_path)

    # Get unique class names
    class_names = df['class'].unique()

    # Plot AUROC curves
    plt.figure(figsize=(8, 6))
    for class_name in class_names:
        class_data = df[df['class'] == class_name]
        fpr = class_data['fpr'].values
        tpr = class_data['tpr'].values
        auc_value = class_data['auc'].values[0]  # Same AUC value repeated for this class
        plt.plot(fpr, tpr, label=f"{class_name} (AUC = {auc_value:.2f})")

    # Plot diagonal line
    plt.plot([0, 1], [0, 1], 'k--', label='Random Guess')

    # Final plot formatting
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title(title)
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.grid(True)
    plt.legend(loc="lower right")
    plt.tight_layout()
    plt.savefig(save_path)
    plt.close()

    print(f"AUROC plot saved to: {save_path}")

In [28]:
plot_auroc_from_csv("ECGOnly_auroc_data.csv", save_path="restored_auroc_plot.png")


AUROC plot saved to: restored_auroc_plot.png
